# Lab 04: Callbacks & Streaming

**Goal:** control how agent output reaches your UI, using callback handlers (sync) and `stream_async` (async iterator).

Course module: `samples/04-callbacks-streaming`

In [ ]:
# Run this first in every lab: puts the repo root on sys.path and loads lab_config
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from lab_config import get_model, describe
print(describe())

In [ ]:
from strands import Agent, tool

@tool
def calculator(a: float, b: float, operation: str = "add") -> str:
    """Perform a math operation on two numbers.

    Args:
        a: First number
        b: Second number
        operation: One of "add", "subtract", "multiply", "divide", "power"
    """
    ops = {
        "add": lambda: a + b, "subtract": lambda: a - b, "multiply": lambda: a * b,
        "divide": lambda: a / b if b else "Error: division by zero", "power": lambda: a ** b,
    }
    return str(ops.get(operation, lambda: f"Unknown operation: {operation}")())

## 1. Default: streaming to stdout
All the agents so far used the default callback handler, which prints text chunks as they arrive.

In [ ]:
result = Agent(model=get_model(), tools=[calculator])("What is 1024 * 768?")

## 2. A custom callback handler
The handler is called with keyword arguments for **every** event. Common keys:

| key | meaning |
|---|---|
| `data` | a streamed text chunk |
| `current_tool_use` | tool call being assembled (name, input) |
| `message` | a complete message (assistant or tool result) |
| `result` | the final `AgentResult` |

This handler buffers output and prints only complete assistant messages plus a line for each tool call.

In [ ]:
def buffered_handler(**kwargs):
    if "message" in kwargs:
        msg = kwargs["message"]
        for block in msg.get("content", []):
            if msg["role"] == "assistant" and "text" in block:
                print("🤖", block["text"].strip())
            if "toolUse" in block:
                print(f"🔧 {block['toolUse']['name']}({block['toolUse']['input']})")

agent = Agent(model=get_model(), tools=[calculator], callback_handler=buffered_handler)
result = agent("What is 2 to the power of 16, minus 1?")

## 3. Silent mode
`callback_handler=None` prints nothing. Use the returned result instead, which is what you want in batch jobs and APIs.

In [ ]:
agent = Agent(model=get_model(), tools=[calculator], callback_handler=None)
result = agent("What is 42 * 42?")
print("Captured:", str(result).strip())

## 4. Async streaming with `stream_async`
The same events arrive as an async generator, which suits FastAPI, websockets, or any async server. Jupyter supports top-level `await`, so you don't need `asyncio.run()` here.

In [ ]:
agent = Agent(model=get_model(), tools=[calculator], callback_handler=None)

last_tool = None
async for event in agent.stream_async("What is 256 + 256? Then divide that by 8."):
    if "data" in event:
        print(event["data"], end="", flush=True)
    elif "current_tool_use" in event and event["current_tool_use"].get("name"):
        name = event["current_tool_use"]["name"]
        if name != last_tool:
            last_tool = name
            print(f"\n🔧 [{name}]", flush=True)
    elif "result" in event:
        print("\n✅ Stream complete:", event["result"].stop_reason)

## 5. Streaming over HTTP with FastAPI (reference)
The course's `fastapi_streaming.py` pattern. Run it as a script (`uv pip install fastapi uvicorn`), not in the notebook:

```python
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from strands import Agent

app = FastAPI()

@app.post("/chat")
async def chat(prompt: str):
    agent = Agent(callback_handler=None)          # one agent per request
    async def gen():
        async for event in agent.stream_async(prompt):
            if "data" in event:
                yield event["data"]
    return StreamingResponse(gen(), media_type="text/plain")
```

## 🧪 Exercise
Write a callback handler that counts streamed chunks and tool calls, then prints a summary line when it receives `result`.

**Next:** [Lab 05: Hooks](05-hooks.ipynb)

In [ ]:
# your code here